In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_recall_curve

print("Day 5: Libraries imported successfully for Threshold Tuning.")


Day 5: Libraries imported successfully for Threshold Tuning.


In [2]:
val_hybrid_risk=np.load(os.path.join("scaled","val_hybrid_risk.npy"))
y_val=np.load(os.path.join("scaled","y_val.npy"))

### FIND BEST THRESHOLD USING VALIDATION DATA

In [3]:
thresholds=np.arange(0.10,0.91,0.01)
best_threshold = 0
best_f1 = 0
for threshold in thresholds:
    val_pred=(val_hybrid_risk>=threshold).astype(int)
    f1 = f1_score(y_val, val_pred, zero_division=0)
    if f1>best_f1:
        best_f1=f1
        best_threshold=threshold
print("=" * 70)
print("BEST HYBRID THRESHOLD")
print("=" * 70)

print("Best Threshold:", best_threshold)
print("Validation F1 :", best_f1)


BEST HYBRID THRESHOLD
Best Threshold: 0.4999999999999998
Validation F1 : 0.9674777162129607


###  APPLY SAME THRESHOLD TO TEST

In [4]:
test_hybrid_risk = np.load(os.path.join("scaled", "test_hybrid_risk.npy"))
y_test = np.load(os.path.join("scaled", "y_test.npy"))
test_hybrid_pred = (
    test_hybrid_risk >= best_threshold
).astype(int)

print("\nTest Predictions Created.")

print("\nPrediction Distribution:")
print(
    np.unique(
        test_hybrid_pred,
        return_counts=True
    )
)


Test Predictions Created.

Prediction Distribution:
(array([0, 1]), array([461436,   6024]))


In [5]:
# ==========================================
# RISK LEVELS
# ==========================================

def risk_level(score):

    if score < 0.30:
        return "Low"

    elif score < 0.70:
        return "Medium"

    else:
        return "High"


test_risk_levels = np.array([
    risk_level(score)
    for score in test_hybrid_risk
])

print("\nRisk Level Distribution:")
print(
    np.unique(
        test_risk_levels,
        return_counts=True
    )
)


Risk Level Distribution:
(array(['High', 'Low', 'Medium'], dtype='<U6'), array([   270, 441114,  26076]))


In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# ==========================================
# HYBRID MODEL EVALUATION
# ==========================================

print("=" * 70)
print("HYBRID MODEL - TEST RESULTS")
print("=" * 70)

accuracy = accuracy_score(y_test, test_hybrid_pred)
precision = precision_score(y_test, test_hybrid_pred, zero_division=0)
recall = recall_score(y_test, test_hybrid_pred, zero_division=0)
f1 = f1_score(y_test, test_hybrid_pred, zero_division=0)

# Hybrid risk score ko probability/ranking ke taur par ROC-AUC mein use karna
roc_auc = roc_auc_score(y_test, test_hybrid_risk)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC-AUC  :", roc_auc)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_hybrid_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, test_hybrid_pred)

print(cm)

HYBRID MODEL - TEST RESULTS
Accuracy : 0.9991336157104351
Precision: 1.0
Recall   : 0.9370041997200187
F1 Score : 0.9674777162129607
ROC-AUC  : 0.999369709351942

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    461031
           1       1.00      0.94      0.97      6429

    accuracy                           1.00    467460
   macro avg       1.00      0.97      0.98    467460
weighted avg       1.00      1.00      1.00    467460


Confusion Matrix:
[[461031      0]
 [   405   6024]]
